In [6]:
# ============================================================
# ÉTAPE 4 : ENTRAÎNEMENT & COMPARAISON DE 3 MODÈLES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
import joblib
import time
warnings.filterwarnings("ignore")

import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             log_loss, brier_score_loss)
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
CONFIG = {
    "n_trials_xgb"  : 80,
    "n_trials_lgb"  : 80,
    "n_trials_rf"   : 40,
    "cv_folds"      : 5,
    "random_state"  : 42,
    "n_jobs"        : -1,
}

In [ ]:
# ─────────────────────────────────────────────────────────────
# A. CHARGEMENT & SPLIT
# ─────────────────────────────────────────────────────────────

print("="*60)
print("  CHARGEMENT DES DONNÉES")
print("="*60)

features_df = pd.read_parquet("../../data/tennis/atp_features_clean_final.parquet")
features_df = features_df.drop(columns=["diff_last20_won"], errors="ignore")

# ── FIX : tri chronologique indispensable pour TimeSeriesSplit ──
# Sans ce tri, les folds peuvent ne contenir qu'une seule classe
# car les labels 0 et 1 sont groupés par blocs dans le fichier
features_df = features_df.sort_values("year").reset_index(drop=True)

# Vérification que les deux classes sont bien mélangées
print("Distribution label par année :")
print(features_df.groupby("year")["label"].value_counts().unstack())

# Vérification que p1_name et p2_name ne sont pas dans X
X = features_df.drop(columns=["label", "year", "p1_name", "p2_name"], errors="ignore")
y = features_df["label"]

print(f"Features : {X.shape[1]}")
print(f"Colonnes fatigue dans X :")
print([c for c in X.columns if "fatigue" in c or "days_since" in c])

train_idx = features_df["year"] <= 2023
val_idx   = features_df["year"] == 2024
test_idx  = features_df["year"] >= 2025

X_train, y_train = X[train_idx].reset_index(drop=True), y[train_idx].reset_index(drop=True)
X_val,   y_val   = X[val_idx].reset_index(drop=True),   y[val_idx].reset_index(drop=True)
X_test,  y_test  = X[test_idx].reset_index(drop=True),  y[test_idx].reset_index(drop=True)

X_trainval = pd.concat([X_train, X_val], ignore_index=True)
y_trainval = pd.concat([y_train, y_val], ignore_index=True)

print(f"✅ Train      : {len(X_train):,} lignes")
print(f"   Validation : {len(X_val):,}  lignes")
print(f"   Test       : {len(X_test):,}  lignes")
print(f"   Features   : {X_train.shape[1]}")

tscv    = TimeSeriesSplit(n_splits=CONFIG["cv_folds"])
results = {}
models  = {}



  CHARGEMENT DES DONNÉES
Distribution label par année :
label     0     1
year             
2020   1466  1466
2021   2735  2735
2022   2918  2918
2023   2995  2995
2024   3159  3159
2025   2286  2861
2026    802   227
✅ Train      : 20,228 lignes
   Validation : 6,318  lignes
   Test       : 6,176  lignes
   Features   : 84


In [3]:
# ─────────────────────────────────────────────────────────────
# UTILITAIRE : évaluation complète
# ─────────────────────────────────────────────────────────────

def evaluate_model(name, model, X_tr, y_tr, X_v, y_v, X_te, y_te):
    res = {"name": name}
    for split_name, Xs, ys in [("train", X_tr, y_tr),
                                ("val",   X_v,  y_v),
                                ("test",  X_te, y_te)]:
        preds  = model.predict(Xs)
        probas = model.predict_proba(Xs)[:, 1]
        res[f"{split_name}_acc"]     = accuracy_score(ys, preds)
        res[f"{split_name}_auc"]     = roc_auc_score(ys, probas)
        res[f"{split_name}_logloss"] = log_loss(ys, probas)
        res[f"{split_name}_brier"]   = brier_score_loss(ys, probas)
    res["overfit_gap"] = res["train_acc"] - res["test_acc"]

    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    print(f"  {'':20} {'Train':>8} {'Val':>8} {'Test':>8}")
    print(f"  {'Accuracy':20} {res['train_acc']:>8.4f} {res['val_acc']:>8.4f} {res['test_acc']:>8.4f}")
    print(f"  {'ROC-AUC':20} {res['train_auc']:>8.4f} {res['val_auc']:>8.4f} {res['test_auc']:>8.4f}")
    print(f"  {'Log Loss':20} {res['train_logloss']:>8.4f} {res['val_logloss']:>8.4f} {res['test_logloss']:>8.4f}")
    print(f"  {'Brier Score':20} {res['train_brier']:>8.4f} {res['val_brier']:>8.4f} {res['test_brier']:>8.4f}")
    print(f"  {'Overfitting gap':20} {res['overfit_gap']:>8.4f}")
    return res

In [ ]:
# ─────────────────────────────────────────────────────────────
# B. MODÈLE 1 — XGBOOST
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("  MODÈLE 1 : XGBOOST — Optimisation Optuna")
print("="*60)

def objective_xgb(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 300, 2000),
        "max_depth"        : trial.suggest_int("max_depth", 3, 10),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        "min_child_weight" : trial.suggest_int("min_child_weight", 1, 20),
        "gamma"            : trial.suggest_float("gamma", 0, 5),
        "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
        "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
        "max_delta_step"   : trial.suggest_int("max_delta_step", 0, 10),
        "objective"        : "binary:logistic",
        "eval_metric"      : "auc",
        "random_state"     : CONFIG["random_state"],
        "n_jobs"           : CONFIG["n_jobs"],
        "tree_method"      : "hist",
        "verbosity"        : 0,
    }

    auc_scores = []
    for tr_idx, val_idx_cv in tscv.split(X_train):
        X_tr_cv = X_train.iloc[tr_idx]
        y_tr_cv = y_train.iloc[tr_idx]
        X_v_cv  = X_train.iloc[val_idx_cv]
        y_v_cv  = y_train.iloc[val_idx_cv]

        # Skip si une classe est absente dans le fold
        if len(y_tr_cv.unique()) < 2 or len(y_v_cv.unique()) < 2:
            continue

        model = xgb.XGBClassifier(**params, early_stopping_rounds=50)
        model.fit(X_tr_cv, y_tr_cv,
                  eval_set=[(X_v_cv, y_v_cv)],
                  verbose=False)
        preds = model.predict_proba(X_v_cv)[:, 1]
        auc_scores.append(roc_auc_score(y_v_cv, preds))

    return np.mean(auc_scores) if auc_scores else 0.5

study_xgb = optuna.create_study(direction="maximize",
                                 sampler=TPESampler(seed=CONFIG["random_state"]))
t0 = time.time()
study_xgb.optimize(objective_xgb, n_trials=CONFIG["n_trials_xgb"],
                   show_progress_bar=True)
print(f"\n✅ XGBoost terminé en {(time.time()-t0)/60:.1f} min")
print(f"   Meilleur AUC CV  : {study_xgb.best_value:.4f}")
print(f"   Meilleurs params : {study_xgb.best_params}")

best_xgb_params = {
    **study_xgb.best_params,
    "objective"   : "binary:logistic",
    "eval_metric" : "auc",
    "random_state": CONFIG["random_state"],
    "n_jobs"      : CONFIG["n_jobs"],
    "tree_method" : "hist",
    "verbosity"   : 0,
}

neg = (y_trainval == 0).sum()
pos = (y_trainval == 1).sum()
best_xgb_params["scale_pos_weight"] = neg / pos  # rééquilibre les classes

xgb_model = xgb.XGBClassifier(**best_xgb_params, early_stopping_rounds=50)
xgb_model.fit(X_trainval, y_trainval,
              eval_set=[(X_val, y_val)],
              verbose=False)

results["XGBoost"] = evaluate_model(
    "XGBoost", xgb_model,
    X_train, y_train, X_val, y_val, X_test, y_test)
models["XGBoost"] = xgb_model
joblib.dump(xgb_model, "../../models/tennis/model_xgboost.pkl")
print("✅ Sauvegardé → model_xgboost.pkl")

In [ ]:
# ─────────────────────────────────────────────────────────────
# C. MODÈLE 2 — LIGHTGBM
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("  MODÈLE 2 : LIGHTGBM — Optimisation Optuna")
print("="*60)

lgb_callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=-1)
]

def objective_lgb(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 300, 3000),
        "max_depth"        : trial.suggest_int("max_depth", 3, 12),
        "num_leaves"       : trial.suggest_int("num_leaves", 20, 300),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq"   : trial.suggest_int("subsample_freq", 1, 10),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
        "reg_l"
        "ambda"       : trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
        "min_split_gain"   : trial.suggest_float("min_split_gain", 0, 1),
        "objective"        : "binary",
        "metric"           : "auc",
        "boosting_type"    : "gbdt",
        "random_state"     : CONFIG["random_state"],
        "n_jobs"           : CONFIG["n_jobs"],
        "verbose"          : -1,
    }

    auc_scores = []
    for tr_idx, val_idx_cv in tscv.split(X_train):
        X_tr_cv = X_train.iloc[tr_idx]
        y_tr_cv = y_train.iloc[tr_idx]
        X_v_cv  = X_train.iloc[val_idx_cv]
        y_v_cv  = y_train.iloc[val_idx_cv]

        # Skip si une classe est absente dans le fold
        if len(y_tr_cv.unique()) < 2 or len(y_v_cv.unique()) < 2:
            continue

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr_cv, y_tr_cv,
                  eval_set=[(X_v_cv, y_v_cv)],
                  callbacks=lgb_callbacks)
        preds = model.predict_proba(X_v_cv)[:, 1]
        auc_scores.append(roc_auc_score(y_v_cv, preds))

    return np.mean(auc_scores) if auc_scores else 0.5

study_lgb = optuna.create_study(direction="maximize",
                                 sampler=TPESampler(seed=CONFIG["random_state"]))
t0 = time.time()
study_lgb.optimize(objective_lgb, n_trials=CONFIG["n_trials_lgb"],
                   show_progress_bar=True)
print(f"\n✅ LightGBM terminé en {(time.time()-t0)/60:.1f} min")
print(f"   Meilleur AUC CV  : {study_lgb.best_value:.4f}")
print(f"   Meilleurs params : {study_lgb.best_params}")

best_lgb_params = {
    **study_lgb.best_params,
    "objective"    : "binary",
    "metric"       : "auc",
    "boosting_type": "gbdt",
    "random_state" : CONFIG["random_state"],
    "n_jobs"       : CONFIG["n_jobs"],
    "verbose"      : -1,
}

best_lgb_params["is_unbalance"] = True

lgb_model = lgb.LGBMClassifier(**best_lgb_params)
lgb_model.fit(X_trainval, y_trainval,
              eval_set=[(X_val, y_val)],
              callbacks=lgb_callbacks)

results["LightGBM"] = evaluate_model(
    "LightGBM", lgb_model,
    X_train, y_train, X_val, y_val, X_test, y_test)
models["LightGBM"] = lgb_model
joblib.dump(lgb_model, "../../models/tennis/model_lightgbm.pkl")
print("✅ Sauvegardé → model_lightgbm.pkl")

In [ ]:
# ─────────────────────────────────────────────────────────────
# D. MODÈLE 3 — RANDOM FOREST
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("  MODÈLE 3 : RANDOM FOREST — Optimisation Optuna")
print("="*60)

def objective_rf(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 200, 1000),
        "max_depth"        : trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf" : trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features"     : trial.suggest_categorical("max_features",
                                                        ["sqrt","log2",0.3,0.5,0.7]),
        "max_samples"      : trial.suggest_float("max_samples", 0.5, 1.0),
        "class_weight"     : "balanced",
        "random_state"     : CONFIG["random_state"],
        "n_jobs"           : CONFIG["n_jobs"],
    }

    auc_scores = []
    for tr_idx, val_idx_cv in tscv.split(X_train):
        X_tr_cv = X_train.iloc[tr_idx]
        y_tr_cv = y_train.iloc[tr_idx]
        X_v_cv  = X_train.iloc[val_idx_cv]
        y_v_cv  = y_train.iloc[val_idx_cv]

        # Skip si une classe est absente dans le fold
        if len(y_tr_cv.unique()) < 2 or len(y_v_cv.unique()) < 2:
            continue

        imp = SimpleImputer(strategy="median")
        X_tr_imp = imp.fit_transform(X_tr_cv)
        X_v_imp  = imp.transform(X_v_cv)

        model = RandomForestClassifier(**params)
        model.fit(X_tr_imp, y_tr_cv)
        preds = model.predict_proba(X_v_imp)[:, 1]
        auc_scores.append(roc_auc_score(y_v_cv, preds))

    return np.mean(auc_scores) if auc_scores else 0.5

study_rf = optuna.create_study(direction="maximize",
                                sampler=TPESampler(seed=CONFIG["random_state"]))
t0 = time.time()
study_rf.optimize(objective_rf, n_trials=CONFIG["n_trials_rf"],
                  show_progress_bar=True)
print(f"\n✅ Random Forest terminé en {(time.time()-t0)/60:.1f} min")
print(f"   Meilleur AUC CV  : {study_rf.best_value:.4f}")
print(f"   Meilleurs params : {study_rf.best_params}")

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   RandomForestClassifier(
        **study_rf.best_params,
        class_weight="balanced",
        random_state=CONFIG["random_state"],
        n_jobs=CONFIG["n_jobs"]
    ))
])
rf_pipeline.fit(X_trainval, y_trainval)

class PipelineWrapper:
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

rf_model = PipelineWrapper(rf_pipeline)

results["RandomForest"] = evaluate_model(
    "Random Forest", rf_model,
    X_train, y_train, X_val, y_val, X_test, y_test)
models["RandomForest"] = rf_model
joblib.dump(rf_pipeline, "../../models/tennis/model_randomforest.pkl")
print("✅ Sauvegardé → model_randomforest.pkl")

In [ ]:

# ─────────────────────────────────────────────────────────────
# E. COMPARAISON DES 3 MODÈLES
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("  COMPARAISON FINALE DES 3 MODÈLES")
print("="*60)

results_df = pd.DataFrame(results).T
print(results_df[["test_acc","test_auc","test_logloss","test_brier","overfit_gap"]].to_string())

best_model_name = results_df["test_auc"].idxmax()
best_model      = models[best_model_name]
print(f"\n🏆 Meilleur modèle : {best_model_name}")
print(f"   Test AUC        : {results_df.loc[best_model_name, 'test_auc']:.4f}")
print(f"   Test Accuracy   : {results_df.loc[best_model_name, 'test_acc']:.4f}")

best_model_data = {
    "model"        : best_model,
    "model_name"   : best_model_name,
    "feature_names": X.columns.tolist(),
    "results"      : results,
    "best_params"  : (study_xgb.best_params if best_model_name == "XGBoost"
                      else study_lgb.best_params if best_model_name == "LightGBM"
                      else study_rf.best_params),
}
joblib.dump(best_model_data, "../../models/tennis/best_model.pkl")
print(f"✅ Meilleur modèle sauvegardé → best_model.pkl")

In [ ]:
# ─────────────────────────────────────────────────────────────
# F. VISUALISATION
# ─────────────────────────────────────────────────────────────

plt.rcParams.update({
    "figure.facecolor" : "#0f1117",
    "axes.facecolor"   : "#1a1d27",
    "axes.edgecolor"   : "#2e3347",
    "axes.labelcolor"  : "#c9d1d9",
    "axes.titlecolor"  : "#ffffff",
    "xtick.color"      : "#8b949e",
    "ytick.color"      : "#8b949e",
    "text.color"       : "#c9d1d9",
    "grid.color"       : "#2e3347",
    "grid.linestyle"   : "--",
    "grid.alpha"       : 0.5,
    "font.family"      : "monospace",
})

ACCENT = "#00d4ff"; GREEN = "#3fb950"; RED = "#f85149"; ORANGE = "#e3b341"
model_colors = {"XGBoost": ACCENT, "LightGBM": GREEN, "RandomForest": ORANGE}

fig = plt.figure(figsize=(18, 12), facecolor="#0f1117")
fig.suptitle("COMPARAISON DES 3 MODÈLES — ATP TENNIS",
             fontsize=20, fontweight="bold", color="white",
             fontfamily="monospace", y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

model_names = list(results.keys())
colors      = [model_colors[n] for n in model_names]

# F1. Accuracy
ax = fig.add_subplot(gs[0, 0])
x, w = np.arange(len(model_names)), 0.25
for i, (split, color) in enumerate([("train","#444"),("val",ORANGE),("test",GREEN)]):
    vals = [results[m][f"{split}_acc"] for m in model_names]
    ax.bar(x + i*w, vals, width=w, label=split.capitalize(),
           color=color, alpha=0.85, edgecolor="none")
ax.set_xticks(x + w)
ax.set_xticklabels(model_names, fontsize=9)
ax.set_title("Accuracy par split", fontsize=11, pad=10)
ax.set_ylabel("Accuracy")
ax.set_ylim(0.45, 0.85)
ax.legend(fontsize=8)
ax.axhline(0.5, color="white", linestyle="--", alpha=0.3)
ax.grid(axis="x", visible=False)

# F2. ROC-AUC
ax = fig.add_subplot(gs[0, 1])
auc_vals = [results[m]["test_auc"] for m in model_names]
bars = ax.bar(model_names, auc_vals, color=colors, alpha=0.85,
              edgecolor="none", width=0.5)
ax.axhline(0.5, color="white", linestyle="--", alpha=0.3)
for bar, val in zip(bars, auc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{val:.4f}", ha="center", fontsize=10, color="white")
ax.set_title("ROC-AUC (Test)", fontsize=11, pad=10)
ax.set_ylim(0.45, 0.85)
ax.grid(axis="x", visible=False)

# F3. Overfitting gap
ax = fig.add_subplot(gs[0, 2])
gap_vals   = [results[m]["overfit_gap"] for m in model_names]
bar_colors = [RED if v > 0.05 else ORANGE if v > 0.02 else GREEN for v in gap_vals]
bars = ax.bar(model_names, gap_vals, color=bar_colors, alpha=0.85,
              edgecolor="none", width=0.5)
for bar, val in zip(bars, gap_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{val:.4f}", ha="center", fontsize=10, color="white")
ax.set_title("Overfitting gap\n(Train acc - Test acc)", fontsize=11, pad=10)
ax.axhline(0, color="white", linestyle="--", alpha=0.3)
ax.grid(axis="x", visible=False)

# F4. Feature importances
ax = fig.add_subplot(gs[1, 0:2])
if best_model_name in ("XGBoost", "LightGBM"):
    imp = pd.Series(best_model.feature_importances_,
                    index=X.columns).sort_values(ascending=False).head(20)
else:
    imp = pd.Series(rf_pipeline.named_steps["model"].feature_importances_,
                    index=X.columns).sort_values(ascending=False).head(20)
ax.barh(imp.index[::-1], imp.values[::-1],
        color=model_colors[best_model_name], alpha=0.85, edgecolor="none")
ax.set_title(f"Top 20 Feature Importances — {best_model_name}", fontsize=11, pad=10)
ax.set_xlabel("Importance")
ax.tick_params(axis="y", labelsize=8)
ax.grid(axis="y", visible=False)

# F5. Tableau récap
ax = fig.add_subplot(gs[1, 2])
ax.axis("off")
metrics    = ["test_acc","test_auc","test_logloss","test_brier","overfit_gap"]
labels_tab = ["Accuracy","ROC-AUC","Log Loss","Brier Score","Overfit Gap"]
table_data = [[f"{results[m][met]:.4f}" for m in model_names] for met in metrics]
table = ax.table(cellText=table_data, rowLabels=labels_tab, colLabels=model_names,
                 cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.8)
for (r, c), cell in table.get_celld().items():
    cell.set_facecolor("#1a1d27" if r > 0 else "#2e3347")
    cell.set_edgecolor("#2e3347")
    cell.set_text_props(color="white")
ax.set_title("Résumé des métriques (Test)", fontsize=11, pad=20)

plt.savefig("../../visualisation/tennis/model_comparison.png", dpi=150, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("✅ Graphique sauvegardé → model_comparison.png")

In [ ]:
# ─────────────────────────────────────────────────────────────
# G. RÉSUMÉ FINAL
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("  RÉSUMÉ FINAL")
print("="*60)
print(f"  🏆 Meilleur modèle  : {best_model_name}")
print(f"  📊 Test Accuracy    : {results[best_model_name]['test_acc']:.4f}")
print(f"  📈 Test ROC-AUC     : {results[best_model_name]['test_auc']:.4f}")
print(f"  📉 Test Log Loss    : {results[best_model_name]['test_logloss']:.4f}")
print(f"  ⚖️  Overfitting gap  : {results[best_model_name]['overfit_gap']:.4f}")
print("="*60)
print("\nFichiers sauvegardés :")
print("  - best_model.pkl         → meilleur modèle + métadonnées")
print("  - model_xgboost.pkl      → XGBoost")
print("  - model_lightgbm.pkl     → LightGBM")
print("  - model_randomforest.pkl → Random Forest")
print("  - model_comparison.png   → graphique comparatif")
print("\n✅ Prêt pour la prédiction → charge best_model.pkl")

In [5]:
# Exclure 2026 du test (données incomplètes)
# et garder uniquement 2025 comme test set
test_idx = features_df["year"] == 2026

X_test, y_test = X[test_idx].reset_index(drop=True), y[test_idx].reset_index(drop=True)
print(f"Test 2025 uniquement : {len(X_test):,} lignes")
print(f"Label 0 : {(y_test==0).sum()} | Label 1 : {(y_test==1).sum()}")

Test 2025 uniquement : 1,029 lignes
Label 0 : 802 | Label 1 : 227
